<a href="https://colab.research.google.com/github/arpitiitb/sql-agent/blob/main/02_text2sql_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agents, Part 2: a text-to-SQL agent, measured

**Goal:** a text-to-SQL agent over a small synthetic insurance database, **measured** on a 20-question test set, with three versions compared:

| Version | What it adds | Idea it teaches |
| --- | --- | --- |
| v0 Baseline | One LLM call, given only the table definitions | How far a plain prompt gets you |
| v1 Context | Column descriptions + business glossary + 3 worked examples (few-shot) | Most text-to-SQL errors are missing context, not bad SQL |
| v2 Agent | Tools to run SQL, see errors and results, fix itself, then submit | When an agent loop is worth its extra cost |
|v3–v5 (rules and a semantic view) |  added after error analysis; see section 9."

**How we score:** *execution accuracy*. Run the model's SQL and a reference SQL, and check that the results match. This is the standard text-to-SQL metric, the same idea as your RAG answer evaluation.

**Cost:** about 20-40 cents with `gpt-4o-mini` (60 test runs, the agent uses several calls each). No Google Drive needed. Synthetic data only, so the whole notebook is safe to commit.

## 1. Setup

In [1]:
%pip install -q -U openai
import os, sys, json, re, time, random, sqlite3
import pandas as pd
from openai import OpenAI
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL = "gpt-4o-mini"
print("Ready")

Ready


## 2. Build the synthetic insurance database

Three tables: `policies` (500 rows), `claims` (466 rows) and `adjusters` (12 rows). The seed is fixed, so everytime gets the same data and the same reference answers.

In [2]:
import random, sqlite3, pandas as pd
def build_db():
    random.seed(7)
    states=["TX","CA","FL","NY","IL"]; products=["auto","home"]
    regions={"TX":"South","FL":"South","CA":"West","NY":"East","IL":"Central"}
    adjusters=[]
    names=["Asha","Ben","Chen","Divya","Eli","Fatima","Gus","Jennie","Ivan","Jaya","Kofi","Lena"]
    for i,n in enumerate(names,1):
        adjusters.append({"adjuster_id":i,"name":n,"region":["South","West","East","Central"][i%4],"years_experience":random.randint(1,15)})
    policies=[]
    for p in range(1,501):
        prod=random.choice(products); st=random.choice(states)
        policies.append({"policy_id":p,"state":st,"product":prod,
            "start_date":f"202{random.randint(3,5)}-{random.randint(1,12):02d}-01",
            "annual_premium":round((3000 if prod=="auto" else 5500)*random.uniform(0.6,1.8),2)})
    claims=[]; cid=1
    for pol in policies:
        for _ in range(random.choices([0,1,2,3],[0.35,0.4,0.18,0.07])[0]):
            prod=pol["product"]
            ctype=random.choice(["collision","theft","injury"] if prod=="auto" else ["hail","water","fire"])
            base={"collision":1000,"theft":1500,"injury":3800,"hail":2200,"water":1800,"fire":7500}[ctype]
            att=int(random.random()<(0.35 if ctype=="injury" else 0.06))
            cost=round(base*random.uniform(0.3,2.2)*(1.6 if att else 1),2)
            reg=regions[pol["state"]]
            adj=random.choice([a["adjuster_id"] for a in adjusters if a["region"]==reg])
            claims.append({"claim_id":cid,"policy_id":pol["policy_id"],"claim_type":ctype,
                "report_date":f"2026-{random.randint(1,10):02d}-{random.randint(1,28):02d}",
                "status":random.choices(["open","closed","reopened"],[0.3,0.62,0.08])[0],
                "attorney":att,"cost_usd":cost,"adjuster_id":adj})
            cid+=1
    conn=sqlite3.connect(":memory:")
    pd.DataFrame(policies).to_sql("policies",conn,index=False)
    pd.DataFrame(claims).to_sql("claims",conn,index=False)
    pd.DataFrame(adjusters).to_sql("adjusters",conn,index=False)
    return conn

conn = build_db()
for t in ["policies", "claims", "adjusters"]:
    print(t, pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn).n[0], "rows")
pd.read_sql("SELECT * FROM claims LIMIT 5", conn)

policies 500 rows
claims 466 rows
adjusters 12 rows


,claim_id,policy_id,claim_type,report_date,status,attorney,cost_usd,adjuster_id
0,1,2,injury,2026-02-17,open,0,4316.06,12
1,2,3,injury,2026-06-05,closed,0,4148.95,9
2,3,3,injury,2026-02-24,closed,0,3138.12,9
3,4,3,theft,2026-03-14,closed,0,2247.40,9
4,5,5,injury,2026-03-14,closed,1,6415.84,12


## 3. The test set and the scorer

20 business questions, each with a **reference SQL** written and checked by hand. Read a few: several need a business definition (what counts as "open", "litigated", "loss ratio", "frequency"), which the model cannot guess from column names alone.

**Scoring rules (execution accuracy):**
- Run both queries. Same number of rows, and each reference row's values appear in a matching predicted row.
- Numbers are compared after rounding to 1 decimal, so 0.7474 and 0.75 count as the same.
- Extra columns in the prediction are allowed (for example the name *and* the count).
- An SQL error is a fail.

This scorer is deliberately simple. Some "fails" will be correct answers in a different shape; you will check those by hand in the error analysis, just like the label errors you found in your RAG project.

In [31]:
TESTS = [
 ("How many claims are there in total?", "SELECT COUNT(*) FROM claims"),
 ("How many policies are there in each state?", "SELECT state, COUNT(*) FROM policies GROUP BY state"),
 ("What is the average claim cost for injury claims?", "SELECT ROUND(AVG(cost_usd),2) FROM claims WHERE claim_type='injury'"),
 ("How many open claims are there?", "SELECT COUNT(*) FROM claims WHERE status IN ('open','reopened')"),
 ("What share of injury claims are litigated, as a percentage?", "SELECT ROUND(100.0*AVG(attorney),1) FROM claims WHERE claim_type='injury'"),
 ("What is the total claim cost by product?", "SELECT p.product, ROUND(SUM(c.cost_usd),2) FROM claims c JOIN policies p ON c.policy_id=p.policy_id GROUP BY p.product"),
 ("What is the claim severity for each claim type?", "SELECT claim_type, ROUND(AVG(cost_usd),2) FROM claims GROUP BY claim_type"),
 ("Which state has the highest total claim cost?", "SELECT p.state FROM claims c JOIN policies p ON c.policy_id=p.policy_id GROUP BY p.state ORDER BY SUM(c.cost_usd) DESC LIMIT 1"),
 ("What is the loss ratio for each product?", "SELECT p.product, ROUND(SUM(COALESCE(c.cost_usd,0))/SUM(p.annual_premium),3) FROM policies p LEFT JOIN (SELECT policy_id, SUM(cost_usd) AS cost_usd FROM claims GROUP BY policy_id) c ON p.policy_id=c.policy_id GROUP BY p.product"),
 ("How many policies have never had a claim?", "SELECT COUNT(*) FROM policies WHERE policy_id NOT IN (SELECT policy_id FROM claims)"),
 ("What is the claim frequency for auto policies?", "SELECT ROUND(1.0*(SELECT COUNT(*) FROM claims c JOIN policies p ON c.policy_id=p.policy_id WHERE p.product='auto')/(SELECT COUNT(*) FROM policies WHERE product='auto'),3)"),
 ("How many claims were reported each month in 2026?", "SELECT substr(report_date,1,7), COUNT(*) FROM claims GROUP BY substr(report_date,1,7)"),
 ("Which adjuster handles the most open claims?", "SELECT a.name FROM claims c JOIN adjusters a ON c.adjuster_id=a.adjuster_id WHERE c.status IN ('open','reopened') GROUP BY a.name ORDER BY COUNT(*) DESC LIMIT 1"),
 ("What is the average cost of litigated versus non-litigated injury claims?", "SELECT attorney, ROUND(AVG(cost_usd),2) FROM claims WHERE claim_type='injury' GROUP BY attorney"),
 ("List the 3 most expensive fire claims with their state.", "SELECT c.claim_id, p.state, c.cost_usd FROM claims c JOIN policies p ON c.policy_id=p.policy_id WHERE c.claim_type='fire' ORDER BY c.cost_usd DESC LIMIT 3"),
 ("How many claims does each region's adjusters handle?", "SELECT a.region, COUNT(*) FROM claims c JOIN adjusters a ON c.adjuster_id=a.adjuster_id GROUP BY a.region"),
 ("What is the total premium of home policies in Florida?", "SELECT ROUND(SUM(annual_premium),2) FROM policies WHERE product='home' AND state='FL'"),
 ("Which claim type has the highest litigation rate?", "SELECT claim_type FROM claims GROUP BY claim_type ORDER BY AVG(attorney) DESC LIMIT 1"),
 ("How many policies have more than one claim?", "SELECT COUNT(*) FROM (SELECT policy_id FROM claims GROUP BY policy_id HAVING COUNT(*) > 1)"),
 ("What is the average claim cost handled by adjusters with more than 10 years of experience?", "SELECT ROUND(AVG(c.cost_usd),2) FROM claims c JOIN adjusters a ON c.adjuster_id=a.adjuster_id WHERE a.years_experience > 10"),
]

def execute(sql):
    return pd.read_sql_query(sql, conn)

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def _same(a, b):
    x, y = _num(a), _num(b)
    if x is not None and y is not None:
        return abs(x - y) <= 0.005 * max(1.0, abs(y))     # within 0.5%
    return str(a).strip().lower() == str(b).strip().lower()

def _row_contains(pred_row, ref_row):
    pool = list(pred_row)
    for need in ref_row:
        for i, have in enumerate(pool):
            if _same(have, need):
                pool.pop(i)
                break
        else:
            return False
    return True

def results_match(pred_sql, ref_sql):
    try:
        pred = execute(pred_sql)
    except Exception as e:
        return False, f"SQL error: {e}"
    ref = execute(ref_sql)
    if len(pred) != len(ref):
        return False, f"row count {len(pred)} vs {len(ref)}"
    pred_rows = [list(r) for r in pred.itertuples(index=False)]
    used = set()
    for r in ref.itertuples(index=False):
        for i, pr in enumerate(pred_rows):
            if i not in used and _row_contains(pr, list(r)):
                used.add(i)
                break
        else:
            return False, f"no predicted row matches reference row {list(r)}"
    return True, "match"

# sanity check: every reference query matches itself
print(all(results_match(s, s)[0] for _, s in TESTS), "- reference queries all run and self-match")

True - reference queries all run and self-match


## 4. v0 Baseline: one call, table definitions only

The model sees only `CREATE TABLE` statements and the question, and returns one SQL query. Nothing runs during generation.

In [4]:
RAW_SCHEMA = "\n".join(r[0] for r in conn.execute("SELECT sql FROM sqlite_master WHERE type='table'"))
print(RAW_SCHEMA)

def extract_sql(text):
    m = re.search(r"```(?:sql)?\s*(.*?)```", text or "", re.S | re.I)
    return (m.group(1) if m else (text or "")).strip().rstrip(";")

def ask_once(question, system):
    r = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": question}])
    return extract_sql(r.choices[0].message.content), r.usage.total_tokens

SYSTEM_V0 = ("Write one SQLite query that answers the question. Return only the SQL in a ```sql block.\n\n"
             "Schema:\n" + RAW_SCHEMA)

def evaluate(fn, label):
    rows = []
    for q, ref in TESTS:
        t0 = time.time()
        out = fn(q)
        sql, tokens = out[0], out[1]
        steps = out[2] if len(out) > 2 else 1
        ok, why = results_match(sql, ref)
        rows.append({"version": label, "question": q, "correct": ok, "why": why, "sql": sql,
                     "tokens": tokens, "steps": steps, "seconds": round(time.time() - t0, 1)})
    df = pd.DataFrame(rows)
    print(f"{label}: {df.correct.sum()}/{len(df)} correct ({100*df.correct.mean():.0f}%), "
          f"avg tokens {df.tokens.mean():.0f}, avg steps {df.steps.mean():.1f}")
    return df

res_v0 = evaluate(lambda q: ask_once(q, SYSTEM_V0), "v0 baseline")
res_v0[~res_v0.correct][["question", "why", "sql"]]

CREATE TABLE "policies" (
"policy_id" INTEGER,
  "state" TEXT,
  "product" TEXT,
  "start_date" TEXT,
  "annual_premium" REAL
)
CREATE TABLE "claims" (
"claim_id" INTEGER,
  "policy_id" INTEGER,
  "claim_type" TEXT,
  "report_date" TEXT,
  "status" TEXT,
  "attorney" INTEGER,
  "cost_usd" REAL,
  "adjuster_id" INTEGER
)
CREATE TABLE "adjusters" (
"adjuster_id" INTEGER,
  "name" TEXT,
  "region" TEXT,
  "years_experience" INTEGER
)
v0 baseline: 13/20 correct (65%), avg tokens 226, avg steps 1.0


,question,why,sql
3,How many open claims are there?,no predicted row matches reference row ['180.0'],SELECT COUNT(*) AS open_claims_count\nFROM cla...
4,"What share of injury claims are litigated, as ...",no predicted row matches reference row ['32.6'],SELECT \n (SUM(CASE WHEN status = 'Litigate...
8,What is the loss ratio for each product?,no predicted row matches reference row ['auto'...,"SELECT \n p.product,\n SUM(c.cost_usd) A..."
13,What is the average cost of litigated versus n...,row count 0 vs 2,SELECT \n CASE \n WHEN claim_type = ...
16,What is the total premium of home policies in ...,no predicted row matches reference row ['23167...,SELECT SUM(annual_premium) AS total_premium\nF...
17,Which claim type has the highest litigation rate?,no predicted row matches reference row ['injury'],"SELECT claim_type, \n COUNT(CASE WHEN st..."
18,How many policies have more than one claim?,row count 119 vs 1,SELECT COUNT(DISTINCT policy_id) AS policy_cou...


**Look at the failures before moving on.** Typical v0 mistakes: treating only `status='open'` as open (missing `reopened`), not knowing what "litigated" means, computing loss ratio the wrong way, or answering frequency as a count. These are **missing knowledge**, not weak SQL.

## 5. v1 Context: descriptions, glossary and few-shot examples

Same single call, but the prompt now carries what a new analyst would be told on day one:
- **Column descriptions**: units, allowed values, how tables join.
- **Business glossary**: the company's definitions of its terms.
- **3 worked examples** (few-shot), chosen to show patterns (joins, rates, groupings) without copying any test question.

In [21]:
SCHEMA_DOC = '''Tables (SQLite):
policies(policy_id INTEGER primary key, state TEXT [TX, CA, FL, NY, IL], product TEXT [auto, home],
         start_date TEXT YYYY-MM-DD, annual_premium REAL in USD)
claims(claim_id INTEGER primary key, policy_id INTEGER -> policies.policy_id,
       claim_type TEXT [collision, theft, injury] for auto, [hail, water, fire] for home,
       report_date TEXT YYYY-MM-DD (all in 2026), status TEXT [open, closed, reopened],
       attorney INTEGER 1 if an attorney is involved else 0, cost_usd REAL in USD,
       adjuster_id INTEGER -> adjusters.adjuster_id)
adjusters(adjuster_id INTEGER primary key, name TEXT, region TEXT [South, West, East, Central],
          years_experience INTEGER)
A claim's state and product come from its policy (join claims to policies).'''

GLOSSARY = '''Business glossary:
- Open claim: status is 'open' OR 'reopened'.
- Litigated claim: attorney = 1. Litigation rate = share of claims with attorney = 1.
- Severity: average cost_usd per claim.
- Frequency: number of claims divided by number of policies.
- Loss ratio: total claim cost_usd divided by total annual_premium (over all policies in the group, including those with no claims).
- Loss ratio and frequency are plain ratios (for example 0.75), not percentages.
- Percentages: only when the question asks for a percentage or share, return 0-100 values rounded to 1 decimal.'''

FEW_SHOT = '''Examples:
Q: How many closed claims are there for home policies?
SQL: SELECT COUNT(*) FROM claims c JOIN policies p ON c.policy_id = p.policy_id WHERE c.status = 'closed' AND p.product = 'home'
Q: What is the litigation rate by state, as a percentage?
SQL: SELECT p.state, ROUND(100.0 * AVG(c.attorney), 1) FROM claims c JOIN policies p ON c.policy_id = p.policy_id GROUP BY p.state
Q: Which adjuster has the highest average claim cost?
SQL: SELECT a.name FROM claims c JOIN adjusters a ON c.adjuster_id = a.adjuster_id GROUP BY a.name ORDER BY AVG(c.cost_usd) DESC LIMIT 1'''

SYSTEM_V1 = ("You write SQLite queries for an insurance analytics team. Return only one query in a ```sql block.\n\n"
             + SCHEMA_DOC + "\n\n" + GLOSSARY + "\n\n" + FEW_SHOT)

res_v1 = evaluate(lambda q: ask_once(q, SYSTEM_V1), "v1 context")
res_v1[~res_v1.correct][["question", "why", "sql"]]

v1 context: 16/20 correct (80%), avg tokens 558, avg steps 1.0


,question,why,sql
8,What is the loss ratio for each product?,"no predicted row matches reference row ['auto', '0.7']","SELECT p.product, \n SUM(c.cost_usd) / SUM(p.annual_premium) AS loss_ratio\nFROM policies p \nLEFT JOIN claims c ON p.policy_id = c.policy_id\nGROUP BY p.product"
10,What is the claim frequency for auto policies?,no predicted row matches reference row ['0.9'],SELECT COUNT(c.claim_id) * 1.0 / COUNT(DISTINCT p.policy_id) AS claim_frequency\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \nWHERE p.product = 'auto'
13,What is the average cost of litigated versus non-litigated injury claims?,row count 1 vs 2,"SELECT \n CASE WHEN c.attorney = 1 THEN 'Litigated' ELSE 'Non-Litigated' END AS claim_type,\n AVG(c.cost_usd) AS average_cost\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \n..."
18,How many policies have more than one claim?,row count 119 vs 1,SELECT COUNT(DISTINCT p.policy_id) \nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \nGROUP BY p.policy_id \nHAVING COUNT(c.claim_id) > 1


## 6. v2 Agent: run, check, fix, submit

Now the model gets two tools, as in Part 1:
- `run_sql(query)`: run a query and see the first rows, or the error. It can explore (for example check the distinct values in a column) and test its query before answering.
- `submit_sql(query)`: hand in the final answer. Calling this ends the loop.

Guardrails are in code: SELECT-only, a 20-row preview, and a step cap.

In [12]:
def run_sql_tool(query):
    q = query.strip().rstrip(";")
    if not q.lower().startswith(("select", "with")) or ";" in q:
        return "ERROR: only a single SELECT query is allowed."
    try:
        df = execute(q)
        return f"{len(df)} rows. First rows:\n" + df.head(20).to_csv(index=False)
    except Exception as e:
        return f"ERROR: {e}"

AGENT_TOOLS = [
    {"type": "function", "function": {"name": "run_sql",
        "description": "Run a read-only SQLite SELECT query and see the first 20 rows or the error.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "submit_sql",
        "description": "Submit the final SQL query that answers the question. Call this exactly once, at the end.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
]
SYSTEM_V2 = (SYSTEM_V1.replace("Return only one query in a ```sql block.", "")
             + "\n\nWork step by step: if unsure about values or joins, use run_sql to check. "
               "Test your final query with run_sql, then call submit_sql with it.")

def sql_agent(question, max_steps=6, verbose=False):
    messages = [{"role": "system", "content": SYSTEM_V2}, {"role": "user", "content": question}]
    tokens = 0
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, temperature=0, messages=messages, tools=AGENT_TOOLS, tool_choice="required")
        tokens += r.usage.total_tokens
        msg = r.choices[0].message
        if not msg.tool_calls:                       # answered in text instead of submitting
            return extract_sql(msg.content), tokens, step
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls]})
        for c in msg.tool_calls:
            args = json.loads(c.function.arguments)
            if c.function.name == "submit_sql":
                if verbose: print(f"[step {step}] SUBMIT: {args['query']}")
                return args["query"].strip().rstrip(";"), tokens, step
            result = run_sql_tool(args["query"])
            if verbose: print(f"[step {step}] run_sql: {args['query']}\n   -> {result[:200]!r}")
            messages.append({"role": "tool", "tool_call_id": c.id, "content": result})
    return "", tokens, max_steps                     # hit the cap without submitting

# Watch one run first
sql_agent("What is the loss ratio for each product?", verbose=True)

[step 1] run_sql: SELECT p.product, SUM(c.cost_usd) AS total_claim_cost, SUM(p.annual_premium) AS total_annual_premium FROM claims c JOIN policies p ON c.policy_id = p.policy_id GROUP BY p.product
   -> '2 rows. First rows:\nproduct,total_claim_cost,total_annual_premium\nauto,731061.05,868071.5\nhome,1191204.87,1495648.55\n'
[step 1] run_sql: SELECT product, SUM(annual_premium) AS total_annual_premium FROM policies GROUP BY product
   -> '2 rows. First rows:\nproduct,total_annual_premium\nauto,978208.22\nhome,1512170.24\n'
[step 2] run_sql: SELECT p.product, SUM(c.cost_usd) AS total_claim_cost, SUM(p.annual_premium) AS total_annual_premium FROM claims c JOIN policies p ON c.policy_id = p.policy_id GROUP BY p.product
   -> '2 rows. First rows:\nproduct,total_claim_cost,total_annual_premium\nauto,731061.05,868071.5\nhome,1191204.87,1495648.55\n'
[step 3] run_sql: SELECT product, SUM(annual_premium) AS total_annual_premium FROM policies GROUP BY product
   -> '2 rows. First rows:\nproduct

('SELECT p.product, ROUND(SUM(c.cost_usd) / SUM(p.annual_premium) * 100, 1) AS loss_ratio FROM claims c JOIN policies p ON c.policy_id = p.policy_id GROUP BY p.product',
 4603,
 5)

In [14]:
res_v2 = evaluate(lambda q: sql_agent(q), "v2 agent")
res_v2[~res_v2.correct][["question", "why", "sql"]]

v2 agent: 17/20 correct (85%), avg tokens 1461, avg steps 2.1


,question,why,sql
8,What is the loss ratio for each product?,"no predicted row matches reference row ['auto', '0.7']","SELECT p.product, ROUND(SUM(c.cost_usd) / SUM(p.annual_premium) * 100, 1) AS loss_ratio FROM claims c JOIN policies p ON c.policy_id = p.policy_id GROUP BY p.product"
10,What is the claim frequency for auto policies?,no predicted row matches reference row ['0.9'],SELECT COUNT(c.claim_id) * 1.0 / COUNT(DISTINCT p.policy_id) AS claim_frequency FROM claims c JOIN policies p ON c.policy_id = p.policy_id WHERE p.product = 'auto'
13,What is the average cost of litigated versus non-litigated injury claims?,"no predicted row matches reference row ['0.0', '5027.9']","SELECT 'Litigated' AS claim_type, AVG(c.cost_usd) AS avg_cost FROM claims c JOIN policies p ON c.policy_id = p.policy_id WHERE c.claim_type = 'injury' AND c.attorney = 1 GROUP BY claim_type UNION ..."


In [22]:
EXTRA_RULES = '''SQL rules:
- Policy-level metrics (frequency, loss ratio, premium totals) must include policies with no claims:
  start FROM policies and LEFT JOIN claims aggregated per policy first.
- Never SUM(annual_premium) after joining policies to individual claims: it repeats the premium per claim.
- "How many X have ..." returns ONE number: count inside a subquery.
- "A versus B" comparisons return one row per group (GROUP BY).'''
SYSTEM_V3 = SYSTEM_V1 + "\n\n" + EXTRA_RULES
res_v3 = evaluate(lambda q: ask_once(q, SYSTEM_V3), "v3 context + rules")
SYSTEM_V2 = (SYSTEM_V3.replace("Return only one query in a ```sql block.", "")
             + "\n\nWork step by step: if unsure, use run_sql to check. Test your final query, then call submit_sql.")
res_v4 = evaluate(lambda q: sql_agent(q), "v4 agent + rules")

v3 context + rules: 16/20 correct (80%), avg tokens 651, avg steps 1.0
v4 agent + rules: 17/20 correct (85%), avg tokens 1594, avg steps 2.0


In [26]:
conn.execute('''CREATE VIEW IF NOT EXISTS policy_summary AS
SELECT p.policy_id, p.state, p.product, p.annual_premium,
       COUNT(c.claim_id) AS claim_count, COALESCE(SUM(c.cost_usd), 0) AS claim_cost
FROM policies p LEFT JOIN claims c ON p.policy_id = c.policy_id
GROUP BY p.policy_id, p.state, p.product, p.annual_premium''')

VIEW_DOC = '''Trusted view (use it for any policy-level metric):
policy_summary(policy_id, state, product, annual_premium, claim_count, claim_cost) -- one row per policy, includes policies with no claims.
- Loss ratio = SUM(claim_cost) / SUM(annual_premium) FROM policy_summary
- Frequency = SUM(claim_count) * 1.0 / COUNT(*) FROM policy_summary'''

SYSTEM_V5 = SYSTEM_V3 + "\n\n" + VIEW_DOC
res_v5 = evaluate(lambda q: ask_once(q, SYSTEM_V5), "v5 context + semantic view")
res_v5[~res_v5.correct][["question", "why", "sql"]]

v5 context + semantic view: 18/20 correct (90%), avg tokens 727, avg steps 1.0


,question,why,sql
13,What is the average cost of litigated versus non-litigated injury claims?,row count 1 vs 2,"SELECT \n CASE WHEN c.attorney = 1 THEN 'Litigated' ELSE 'Non-Litigated' END AS claim_type,\n AVG(c.cost_usd) AS average_cost\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \n..."
14,List the 3 most expensive fire claims with their state.,"no predicted row matches reference row ['384.0', 'ny', '25126.1']","SELECT p.state, c.cost_usd \nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \nWHERE c.claim_type = 'fire' \nORDER BY c.cost_usd DESC \nLIMIT 3"


## 7. Compare the five versions

Accuracy is what users feel; tokens and seconds are what the business pays.

In [33]:
allres = pd.concat([res_v0, res_v1, res_v2, res_v3, res_v4, res_v5])
summary = allres.groupby("version", sort=False).agg(
    correct=("correct", "sum"), accuracy_pct=("correct", lambda s: round(100 * s.mean())),
    avg_tokens=("tokens", "mean"), avg_steps=("steps", "mean"), avg_seconds=("seconds", "mean")).round(1)
summary["of"] = len(TESTS)
summary

,correct,accuracy_pct,avg_tokens,avg_steps,avg_seconds,of
version,,,,,,
v0 baseline,13,65,226.0,1.0,1.3,20
v1 context,16,80,557.6,1.0,1.1,20
v2 agent,17,85,1461.3,2.1,2.7,20
v3 context + rules,16,80,650.9,1.0,1.0,20
v4 agent + rules,17,85,1594.1,2.0,2.5,20
v5 context + semantic view,18,90,727.3,1.0,1.0,20


## 8. Error analysis

For each remaining failure, look at the SQL and decide which bucket it belongs to:
1. **Scorer too strict**: the answer is right but in another shape (for example labels instead of 0/1). Count it as correct by hand and note it.
2. **Ambiguous question**: a reasonable reading differs from the reference. Fix the question wording or the glossary.
3. **Missing knowledge**: a definition or join rule the prompt does not contain. Add it to the glossary.
4. **Real SQL mistake**: wrong join, wrong grouping, wrong filter.

In [34]:
fails = allres[~allres.correct][["version", "question", "why", "sql"]]
pd.set_option("display.max_colwidth", 200)
fails

,version,question,why,sql
3,v0 baseline,How many open claims are there?,no predicted row matches reference row [180],SELECT COUNT(*) AS open_claims_count\nFROM claims\nWHERE status = 'open'
4,v0 baseline,"What share of injury claims are litigated, as a percentage?",no predicted row matches reference row [32.6],SELECT \n (SUM(CASE WHEN status = 'Litigated' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS litigated_percentage\nFROM \n claims\nWHERE \n claim_type = 'Injury'
8,v0 baseline,What is the loss ratio for each product?,"no predicted row matches reference row ['auto', 0.747]","SELECT \n p.product,\n SUM(c.cost_usd) AS total_claims,\n SUM(p.annual_premium) AS total_premiums,\n CASE \n WHEN SUM(p.annual_premium) > 0 THEN SUM(c.cost_usd) / SUM(p.annual_p..."
13,v0 baseline,What is the average cost of litigated versus non-litigated injury claims?,row count 0 vs 2,"SELECT \n CASE \n WHEN claim_type = 'litigated' THEN 'Litigated' \n ELSE 'Non-Litigated' \n END AS claim_category,\n AVG(cost_usd) AS average_cost\nFROM claims\nWHERE claim_..."
16,v0 baseline,What is the total premium of home policies in Florida?,no predicted row matches reference row [231675.88],SELECT SUM(annual_premium) AS total_premium\nFROM policies\nWHERE product = 'home' AND state = 'Florida'
17,v0 baseline,Which claim type has the highest litigation rate?,no predicted row matches reference row ['injury'],"SELECT claim_type, \n COUNT(CASE WHEN status = 'Litigated' THEN 1 END) * 1.0 / COUNT(*) AS litigation_rate\nFROM claims\nGROUP BY claim_type\nORDER BY litigation_rate DESC\nLIMIT 1"
18,v0 baseline,How many policies have more than one claim?,row count 119 vs 1,SELECT COUNT(DISTINCT policy_id) AS policy_count\nFROM claims\nGROUP BY policy_id\nHAVING COUNT(claim_id) > 1
8,v1 context,What is the loss ratio for each product?,"no predicted row matches reference row ['auto', 0.747]","SELECT p.product, \n SUM(c.cost_usd) / SUM(p.annual_premium) AS loss_ratio\nFROM policies p \nLEFT JOIN claims c ON p.policy_id = c.policy_id\nGROUP BY p.product"
10,v1 context,What is the claim frequency for auto policies?,no predicted row matches reference row [0.888],SELECT COUNT(c.claim_id) * 1.0 / COUNT(DISTINCT p.policy_id) AS claim_frequency\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \nWHERE p.product = 'auto'
13,v1 context,What is the average cost of litigated versus non-litigated injury claims?,row count 1 vs 2,"SELECT \n CASE WHEN c.attorney = 1 THEN 'Litigated' ELSE 'Non-Litigated' END AS claim_type,\n AVG(c.cost_usd) AS average_cost\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \n..."


In [35]:
# Write your manual verdicts here, then re-compute an "adjusted" accuracy for the README.
# Example: manual = {("v2 agent", "What is the average cost of litigated versus non-litigated injury claims?"): "scorer too strict"}
FIRE = "List the 3 most expensive fire claims with their state."
LIT = "What is the average cost of litigated versus non-litigated injury claims?"
manual = {(v, FIRE): "scorer too strict" for v in ["v3 context + rules", "v4 agent + rules"]}
manual[("v2 agent", LIT)] = "scorer too strict"
allres["bucket"] = [manual.get((v, q), "" if ok else "unreviewed") for v, q, ok in zip(allres.version, allres.question, allres.correct)]
allres["adjusted_correct"] = allres.correct | (allres.bucket == "scorer too strict")
print(allres.groupby("version", sort=False).adjusted_correct.sum())
allres.to_csv("text2sql_results.csv", index=False)
print("Saved text2sql_results.csv (safe to commit: synthetic data only)")

version
v0 baseline                   13
v1 context                    16
v2 agent                      18
v3 context + rules            17
v4 agent + rules              18
v5 context + semantic view    18
Name: adjusted_correct, dtype: int64
Saved text2sql_results.csv (safe to commit: synthetic data only)


In [30]:
HELDOUT = [
 ("What is the loss ratio by state?",
  "SELECT p.state, ROUND(SUM(COALESCE(c.cost,0))/SUM(p.annual_premium),3) FROM policies p LEFT JOIN (SELECT policy_id, SUM(cost_usd) AS cost FROM claims GROUP BY policy_id) c ON p.policy_id=c.policy_id GROUP BY p.state"),
 ("What is the claim frequency for home policies?",
  "SELECT ROUND(1.0*(SELECT COUNT(*) FROM claims c JOIN policies p ON c.policy_id=p.policy_id WHERE p.product='home')/(SELECT COUNT(*) FROM policies WHERE product='home'),3)"),
 ("How many adjusters handle more than 40 claims?",
  "SELECT COUNT(*) FROM (SELECT adjuster_id FROM claims GROUP BY adjuster_id HAVING COUNT(*) > 40)"),
 ("What is the average annual premium of policies that have at least one litigated claim?",
  "SELECT ROUND(AVG(annual_premium),2) FROM policies WHERE policy_id IN (SELECT policy_id FROM claims WHERE attorney=1)"),
 ("Which month in 2026 had the highest total claim cost?",
  "SELECT substr(report_date,1,7) FROM claims GROUP BY substr(report_date,1,7) ORDER BY SUM(cost_usd) DESC LIMIT 1"),
]
MAIN_TESTS = TESTS
TESTS = HELDOUT                     # evaluate() reads the global TESTS
ho = pd.concat([
    evaluate(lambda q: ask_once(q, SYSTEM_V1), "v1 context"),
    evaluate(lambda q: ask_once(q, SYSTEM_V3), "v3 context + rules"),
    evaluate(lambda q: sql_agent(q), "v4 agent + rules"),
    evaluate(lambda q: ask_once(q, SYSTEM_V5), "v5 context + semantic view"),
])
TESTS = MAIN_TESTS                  # restore the main test set
ho.to_csv("text2sql_heldout_results.csv", index=False)
ho[~ho.correct][["version", "question", "why", "sql"]]

v1 context: 2/5 correct (40%), avg tokens 565, avg steps 1.0
v3 context + rules: 3/5 correct (60%), avg tokens 661, avg steps 1.0
v4 agent + rules: 3/5 correct (60%), avg tokens 1886, avg steps 2.2
v5 context + semantic view: 3/5 correct (60%), avg tokens 732, avg steps 1.0


,version,question,why,sql
0,v1 context,What is the loss ratio by state?,"no predicted row matches reference row ['ca', '0.8']","SELECT p.state, \n SUM(c.cost_usd) / SUM(p.annual_premium) AS loss_ratio\nFROM policies p\nLEFT JOIN claims c ON p.policy_id = c.policy_id\nGROUP BY p.state"
1,v1 context,What is the claim frequency for home policies?,no predicted row matches reference row ['1.0'],SELECT COUNT(c.claim_id) * 1.0 / COUNT(DISTINCT p.policy_id) AS claim_frequency\nFROM claims c \nJOIN policies p ON c.policy_id = p.policy_id \nWHERE p.product = 'home'
3,v1 context,What is the average annual premium of policies that have at least one litigated claim?,row count 56 vs 1,SELECT AVG(p.annual_premium) \nFROM policies p \nJOIN claims c ON p.policy_id = c.policy_id \nWHERE c.attorney = 1 \nGROUP BY p.policy_id
0,v3 context + rules,What is the loss ratio by state?,"no predicted row matches reference row ['ca', '0.8']","SELECT p.state, \n SUM(COALESCE(c.cost_usd, 0)) / SUM(p.annual_premium) AS loss_ratio\nFROM policies p\nLEFT JOIN claims c ON p.policy_id = c.policy_id\nGROUP BY p.state"
1,v3 context + rules,What is the claim frequency for home policies?,no predicted row matches reference row ['1.0'],SELECT COUNT(c.claim_id) * 1.0 / COUNT(p.policy_id) AS claim_frequency\nFROM policies p LEFT JOIN claims c ON p.policy_id = c.policy_id\nWHERE p.product = 'home'
0,v4 agent + rules,What is the loss ratio by state?,"no predicted row matches reference row ['ca', '0.8']","SELECT p.state, ROUND(SUM(c.cost_usd) / SUM(p.annual_premium), 4) AS loss_ratio FROM policies p LEFT JOIN claims c ON p.policy_id = c.policy_id GROUP BY p.state"
1,v4 agent + rules,What is the claim frequency for home policies?,no predicted row matches reference row ['1.0'],SELECT COUNT(c.claim_id) * 1.0 / COUNT(p.policy_id) AS claim_frequency \nFROM policies p \nLEFT JOIN claims c ON p.policy_id = c.policy_id \nWHERE p.product = 'home'
0,v5 context + semantic view,What is the loss ratio by state?,"no predicted row matches reference row ['tx', '0.8']","SELECT state, SUM(claim_cost) * 1.0 / SUM(annual_premium) AS loss_ratio \nFROM policy_summary \nGROUP BY state"
3,v5 context + semantic view,What is the average annual premium of policies that have at least one litigated claim?,row count 56 vs 1,SELECT AVG(p.annual_premium) \nFROM policies p \nJOIN claims c ON p.policy_id = c.policy_id \nWHERE c.attorney = 1 \nGROUP BY p.policy_id \nHAVING COUNT(c.claim_id) > 0


In [32]:
REF = dict(MAIN_TESTS + HELDOUT)
for d in [res_v0, res_v1, res_v2, res_v3, res_v4, res_v5, ho]:
    out = [results_match(s, REF[q]) for q, s in zip(d.question, d.sql)]
    d["correct"] = [o[0] for o in out]
    d["why"] = [o[1] for o in out]
print(ho.groupby("version", sort=False).correct.sum())
ho.to_csv("text2sql_heldout_results.csv", index=False)

version
v1 context                    2
v3 context + rules            3
v4 agent + rules              3
v5 context + semantic view    4
Name: correct, dtype: int64


## 9. What we learned

## 9. Results and what I learned

| Version | Dev set (of 20) | Held-out (of 5) | Tokens / question |
| --- | --- | --- | --- |
| v0 Baseline | 13 | – | 226 |
| v1 Context | 16 | 2 | 558 |
| v2 Agent | 18 | – | 1,461 |
| v3 Context + rules | 17 | 3 | 651 |
| v4 Agent + rules | 18 | 3 | 1,594 |
| v5 Context + semantic view | **19** | **4** | 727 |

- **Context was the biggest, cheapest gain (65% → 80%).** The baseline failed on business definitions ("open", "litigated", state codes), not on SQL.
- **The hardest errors ran without error and looked plausible.** Loss ratio and frequency need every policy, including those without claims, and must not repeat premiums once per claim. Prompt rules did not stop this, even on unseen questions.
- **The agent loop added ~2.5x cost for little gain.** Self-checking catches SQL errors and empty results, not wrong logic that looks right.
- **A trusted view (`policy_summary`) fixed the join errors in a single cheap call** and was the only version to get both policy-level metrics right on held-out questions. Put error-prone business logic into trusted views or metrics, not prompts.
- **Prompt changes can regress other answers** (a percentage rule made ratios ×100), so the full set is re-run after every change.
- **Requiring a `submit_sql` tool call** (`tool_choice="required"`) stopped the agent from answering in prose.
- **Limits:** 25 questions is small (one question = 5 points), so 1–2 question differences are noise. Development 95% vs held-out 80% shows some tuning to the development set.
